# 工具call 



In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

@tool # 使用 @tool 装饰器，将 Python 函数转换为 LangChain Tool
def get_weather(location: str): # 定义工具函数 get_weather, location 参数用于接收城市名称
    """Call to get the current weather.""" # 工具函数的 docstring 会被作为工具的描述信息，提供给 LLM
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

@tool
def get_coolest_cities():
    """Get a list of coolest cities"""
    return "nyc, sf"



# 创建 ToolNode 实例，注册工具列表 (包含 get_weather 和 get_coolest_cities 两个工具)
tools = [get_weather, get_coolest_cities]
tool_node = ToolNode(tools)

# 构造包含单个工具调用请求的 AIMessage
message_with_single_tool_call = AIMessage( # 创建 AIMessage
    content="", # content 为空字符串，表示该消息主要用于工具调用，不包含文本内容
    tool_calls=[ # tool_calls 参数，包含工具调用请求列表
        {
            "name": "get_weather", #  工具名称，必须与注册的工具名称一致
            "args": {"location": "sf"}, #  工具参数，必须与工具函数定义的参数匹配
            "id": "tool_call_id", #  工具调用 ID，用于唯一标识工具调用，  可以自定义
            "type": "tool_call", #  消息类型，固定为 "tool_call"
        }
    ],
)

# 手动调用 ToolNode，输入为包含 AIMessage 的状态字典
tool_node_output = tool_node.invoke({"messages": [message_with_single_tool_call]})

print(tool_node_output) # 打印 ToolNode 的输出结果

{'messages': [ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='tool_call_id')]}


In [2]:
# 构造包含多个工具调用请求的 AIMessage
message_with_multiple_tool_calls = AIMessage( # 创建 AIMessage
    content="",
    tool_calls=[ # tool_calls 参数，包含多个工具调用请求
        {
            "name": "get_coolest_cities", # 工具 1：get_coolest_cities
            "args": {},
            "id": "tool_call_id_1",
            "type": "tool_call",
        },
        {
            "name": "get_weather", # 工具 2：get_weather
            "args": {"location": "sf"},
            "id": "tool_call_id_2",
            "type": "tool_call",
        },
    ],
)

# 手动调用 ToolNode,  输入为包含 AIMessage 的状态字典
tool_node_output = tool_node.invoke({"messages": [message_with_multiple_tool_calls]})

print(tool_node_output) # 打印 ToolNode 的输出结果

{'messages': [ToolMessage(content='nyc, sf', name='get_coolest_cities', tool_call_id='tool_call_id_1'), ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='tool_call_id_2')]}


## 工具调用错误处理

demo ：实现了一个具有工具调用能力和降级策略的俳句生成工作流

In [4]:
import json
from typing import Literal

from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.messages.modifier import RemoveMessage

from langgraph.graph import MessagesState, StateGraph, END, START
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

load_dotenv()


class HaikuRequest(BaseModel):
    topic: list[str] = Field(
        max_length=3,
        min_length=3,
    )

# 定义俳句生成工具（使用@tool装饰器标记为工具）
@tool
def master_haiku_generator(request: HaikuRequest):
    """Generates a haiku based on the provided topics."""
    model = ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        base_url=os.getenv("BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0,
    )
    # 创建处理链：模型调用 → 输出解析为字符串
    chain = model | StrOutputParser()
    topics = ", ".join(request.topic)
    haiku = chain.invoke(f"Write a haiku about {topics}")
    return haiku

# 工具调用节点：执行AI模型发起的工具调用
def call_tool(state: MessagesState):
    # 创建工具名称到工具函数的映射字典
    tools_by_name = {master_haiku_generator.name: master_haiku_generator}
    messages = state["messages"]
    last_message = messages[-1]  # 获取最后一条消息
    output_messages = []
    # 遍历最后一条消息中的所有工具调用
    for tool_call in last_message.tool_calls:
        try:
            # 根据工具名称找到对应的工具函数并调用，传入参数
            tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            # 将工具调用结果封装为ToolMessage添加到输出消息列表
            output_messages.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        except Exception as e:
            # 如果工具调用失败，捕获异常并返回错误信息
            # 将错误信息封装为ToolMessage，并在additional_kwargs中标记错误
            output_messages.append(
                ToolMessage(
                    content=str(e), 
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                    additional_kwargs={"error": e},  # 在额外参数中存储错误对象，用于后续错误处理
                )
            )
    return {"messages": output_messages}

# 初始化基础模型（较弱的模型）
model = ChatOpenAI(
    model=os.getenv("MODEL_NAME_2"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)
model_with_tools = model.bind_tools([master_haiku_generator])

# 初始化更强大的模型（用于降级策略）
better_model = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)
better_model_with_tools = better_model.bind_tools([master_haiku_generator])

def should_continue(state: MessagesState):
    # 决定是否继续工具调用循环或结束流程
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:  # 如果最后一条消息包含工具调用请求
        return "tools"  # 继续执行工具调用
    return END  # 否则结束流程

def should_fallback(
    state: MessagesState,
) -> Literal["agent", "remove_failed_tool_call_attempt"]:
    # 决定是否需要降级到更强大的模型
    messages = state["messages"]
    # 查找是否有失败的工具调用消息（通过 additional_kwargs 中的 error 标记识别）
    failed_tool_messages = [
        msg
        for msg in messages
        if isinstance(msg, ToolMessage)
        and msg.additional_kwargs.get("error") is not None
    ]
    if failed_tool_messages:  # 如果存在失败的工具调用
        return "remove_failed_tool_call_attempt"  # 路由到移除失败尝试的节点
    return "agent"  # 否则继续使用当前模型

def call_model(state: MessagesState):
    # 使用基础模型处理消息
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

def remove_failed_tool_call_attempt(state: MessagesState):
    # 移除失败的工具调用尝试，清理消息历史
    messages = state["messages"]
    # 从后向前查找最近的AI消息索引
    last_ai_message_index = next(
        i
        for i, msg in reversed(list(enumerate(messages)))
        if isinstance(msg, AIMessage)
    )
    # 获取需要移除的消息（从最近的AI消息开始的所有消息）
    messages_to_remove = messages[last_ai_message_index:]
    # 返回移除指令，通过RemoveMessage标记需要移除的消息
    return {"messages": [RemoveMessage(id=m.id) for m in messages_to_remove]}

# 降级策略：使用更强大的模型重试
def call_fallback_model(state: MessagesState):
    # 使用更强大的模型处理消息
    messages = state["messages"]
    response = better_model_with_tools.invoke(messages)
    return {"messages": [response]}

# 创建状态图
workflow = StateGraph(MessagesState)

# 添加节点
workflow.add_node("agent", call_model)  # 基础模型节点
workflow.add_node("tools", call_tool)  # 工具调用节点
workflow.add_node("remove_failed_tool_call_attempt", remove_failed_tool_call_attempt)  # 清理失败尝试节点
workflow.add_node("fallback_agent", call_fallback_model)  # 降级模型节点

# 添加边和条件边
workflow.add_edge(START, "agent")  # 流程从agent节点开始
workflow.add_conditional_edges("agent", should_continue, ["tools", END])  # 根据should_continue函数决定是继续工具调用还是结束
# 根据工具调用结果决定是继续使用当前模型还是清理失败尝试
workflow.add_conditional_edges("tools", should_fallback, path_map = {"agent": "agent", "remove_failed_tool_call_attempt": "remove_failed_tool_call_attempt"}) 
workflow.add_edge("remove_failed_tool_call_attempt", "fallback_agent")  # 清理失败尝试后使用降级模型
workflow.add_edge("fallback_agent", "tools")  # 降级模型生成的工具调用请求继续由tools节点处理

app = workflow.compile()

## 工具函数返回 Command 对象，更新图状态

In [ ]:
from typing_extensions import Annotated, Any

from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage

from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt.chat_agent_executor import AgentState
from langgraph.prebuilt import ToolNode, InjectedState


USER_INFO = [ # 定义用户信息列表 (示例数据)
    {"user_id": "1", "name": "张三", "location": "广州, GZ"},
    {"user_id": "2", "name": "李四", "location": "深圳, SZ"},
]

USER_ID_TO_USER_INFO = {info["user_id"]: info for info in USER_INFO} #  用户 ID -> 用户信息 字典

class State(AgentState): # 定义图状态结构体, 继承自 AgentState, 并添加 user_info 状态键
    user_info: dict[str, Any]
    user_id: str

@tool
def lookup_user_info( # 定义工具函数 lookup_user_info
    tool_call_id: Annotated[str, InjectedToolCallId],
    user_id: Annotated[str, InjectedState("user_id")]
):
    """Use this to look up user information to better assist them with their questions.""" # 工具描述信息
    if user_id is None:
        raise ValueError("Please provide user ID")
    if user_id not in USER_ID_TO_USER_INFO:
        raise ValueError(f"User '{user_id}' not found")

    user_info = USER_ID_TO_USER_INFO[user_id] # 根据 user_id 查询用户信息

    return Command( # 工具函数返回 Command 对象
        update={ # Command 对象包含状态更新指令
            "user_info": user_info, # 更新 user_info 状态键，值为查询到的用户信息
            "messages": [ # 更新 messages 状态键，添加 ToolMessage
                ToolMessage(
                    "Successfully looked up user information", tool_call_id=tool_call_id
                )
            ],
        }
    )

# 初始化状态图
graph = StateGraph(State)

# 定义节点
def agent_node(state: State):
    """智能体节点，处理用户请求"""
    messages = state["messages"]
    user_info = state.get("user_info", {})
    
    # 如果有用户信息，将其添加到系统消息中
    if user_info:
        system_message = f"You are assisting {user_info['name']} who lives in {user_info['location']}."
    else:
        system_message = "You are a helpful assistant."
    
    # 调用模型处理请求
    model = ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        base_url=os.getenv("BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0,
    )
    model_with_tools = model.bind_tools([lookup_user_info])
    response = model_with_tools.invoke([{"role": "system", "content": system_message}] + messages)
    return {"messages": [response]}

def should_use_tools(state: State):
    """决定是否使用工具"""
    messages = state["messages"]
    last_message = messages[-1]
    
    # 检查最后一条消息是否包含工具调用
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

# 使用 ToolNode 简化工具调用逻辑
tools_node = ToolNode([lookup_user_info])

# 添加节点到图
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)

# 添加边和条件边
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_conditional_edges("agent", should_use_tools, {"tools": "tools", "end": END})

# 编译图
agent = graph.compile()

# 调用 ReAct 智能体，通过 config 参数传递运行时参数 user_id
for chunk in agent.stream(
    # 通过 user_id 状态键传递运行时参数
    {"messages": [("human", "我是谁？我应该住在哪里?")], "user_id": "1"},
):
    print(chunk)



{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_99f6c9ba4d7e4745bd0169', 'function': {'arguments': '{}', 'name': 'lookup_user_info'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 158, 'total_tokens': 174, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-b081e8bc-9eab-9218-bc5f-8adfaa278f51', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--a0c8e978-f68e-4d00-b124-d3f92e893da4-0', tool_calls=[{'name': 'lookup_user_info', 'args': {}, 'id': 'call_99f6c9ba4d7e4745bd0169', 'type': 'tool_call'}], usage_metadata={'input_tokens': 158, 'output_tokens': 16, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}}
{'tools': {'user_info': {'user_id': '1', 'name': '张三', 'locatio

In [11]:
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass